## Importação da bibliotecas utilizadas.


- `boto3`: Biblioteca oficial da AWS para interagir com serviços como S3, EC2, etc.  
- `botocore`: Biblioteca base do boto3, usada para tratamento de erros e configurações detalhadas.  
- `os`: Biblioteca padrão do Python para acessar recursos do sistema operacional, como variáveis de ambiente.  
- `load_dotenv` (de `dotenv`): Função que carrega variáveis do arquivo `.env` para o ambiente do sistema.


In [ ]:
import boto3
import botocore
import os
from dotenv import load_dotenv

## Definição dos Objetivos

Com base na base de dados, foram definidas as 3 análises principais a serem feitas:

#### 1. Entre pessoas físicas e jurídicas, qual grupo concentra a maior proporção da dívida total e qual é a situação predominante dessas dívidas?

#### 2. Como a quantidade e o valor total das dívidas em execução variam conforme o tempo de atraso, e quais faixas de atraso concentram a maior parte da dívida ativa?

#### 3. Qual é a distribuição da quantidade e do valor das dívidas <u>milionárias</u> entre os tipos de empresa LTDA, S/A e Outros, e como o tempo médio de atraso varia entre esses grupos.

## Definindo credenciais de acesso na plataforma AWS e iniciando uma sessão

Carregar variáveis de ambiente de um arquivo .env

In [ ]:
load_dotenv()

Atribuição das variaveis de acesso a partir da extração das variaveis de ambiente

In [ ]:
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_session_token = os.getenv("AWS_SESSION_TOKEN")
region = os.getenv("AWS_REGION")

Cria uma sessão AWS usando as credenciais anteriores

In [ ]:
session = boto3.Session(
    
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token,
    region_name=region
)


s3 = session.client('s3')

Definição do nome do bucket e do arquivo a ser feito o upload

In [ ]:
bucket_nome = "rhamon-pires-sprint04-bucket"
arquivo_nome = "DEVEDORES_DIVIDA_ATIVA-2024-12.csv" 
key = arquivo_nome     #ja esta no diretorio, mas para manter no formato padronizado, caso estivesse em outro diretorio

## Definição das funções de acesso

Define uma função que verifica se um bucket S3 existe.  
Se ao tentar acessá-lo for lançada a exceção 404 (não encontrado), significa que o bucket não existe, e então ele é criado.

In [ ]:
def criar_bucket(bucket_nome):

    s3 = boto3.client('s3', region_name='us-east-1')

    try:
        s3.head_bucket(Bucket=bucket_nome)
        print(f"Bucket '{bucket_nome}' já existe.")

    except botocore.exceptions.ClientError as e:
        
        error_code = e.response['Error']['Code']

        if error_code in ['404', 'NoSuchBucket']:
            
            s3.create_bucket(Bucket=bucket_nome)
            print(f"Bucket '{bucket_nome}' criado com sucesso.")

        elif error_code in ['403', 'AccessDenied']:
            
            print(f"Você não tem permissão para acessar o bucket '{bucket_nome}'.")

        else:
            raise

Define uma função que recebe o nome do bucket, o nome do arquivo e a key (nesse caso são iguais, mas para manter padronizados os parametros).  
Tenta acessar o objeto; se não existir (exceção lançada), o arquivo é enviado ao bucket com a key informada.

In [ ]:
def upload_csv(bucket_nome, arquivo_nome, key):

    s3 = boto3.client('s3', region_name='us-east-1')
    
    try:
        
        s3.head_object(Bucket=bucket_nome, Key=key)
        print(f"O arquivo '{key}' já existe em '{bucket_nome}'. Upload ignorado.")

    except botocore.exceptions.ClientError as e:

        if e.response['Error']['Code'] == '404':
            
            s3.upload_file(arquivo_nome, bucket_nome, key)
            print(f"Arquivo '{arquivo_nome}' enviado para 's3://{bucket_nome}/{key}'")
            
        else:
            raise

## Chamada das funcões para criação do bucket e envio do CSV

Executa as funções responsáveis por verificar/criar o bucket S3 e, em seguida, enviar o arquivo CSV caso ele ainda não exista.

In [ ]:
criar_bucket(bucket_nome)


In [ ]:
upload_csv(bucket_nome, arquivo_nome, key)